# FINA4030A — Lab 5
## Auditing a model you did not build

**Class 5.** Submit this notebook by 23:59 on **22 October**.

> **Before you type anything: File → Save a copy in Drive.**

You have a three-statement model and a DCF for Texas Instruments, built by an
agent from SEC XBRL data, and **the request that produced it**. Read the request
first. Half of what goes wrong in a model is traceable to what was and was not
asked for, and an audit that never looks at the instruction is an audit of the
wrong thing.

**The model's arithmetic is correct.** Gross profit ties. The pre-tax bridge
ties. PP&E rolls forward. The DCF cross-checks itself in both directions. You
are not looking for a wrong number, and you will lose the session if you start
at A1 and read down.

You are looking for something harder. The workbook has a `Checks` tab that opens
by saying:

> *Every figure below is a live formula. Anything other than "OK" needs looking
> at before the model is used.*

Both halves of that are true. Your job is to find out **what it would take to
make each of those checks fail**, and what it means when the answer turns out to
be "nothing".

**Four passes, in this order.** Reference integrity → hardcodes → missing
conventions → unverified inputs. Find at least one instance of each class, **or
demonstrate that it is absent.** Demonstrating absence is harder than finding
presence, and is marked accordingly.

**What is marked: the evidence, not the assertion.** "The balance check is weak"
scores nothing. "I set FY2026 distributions to \$20bn, total assets went to
−\$37bn, and the balance check still read OK" scores well. Then: what your worst
defect is worth, in dollars per share.


In [ ]:
# Setup. Run this first. The install takes about half a minute.
!pip install formulas -q

REQUIRED_CLIENT = "1.1"
REPO = "https://raw.githubusercontent.com/fy-ericlam/fina4030a/main"

import importlib, sys, urllib.request, warnings
warnings.filterwarnings("ignore")

urllib.request.urlretrieve(f"{REPO}/fina4030a.py", "fina4030a.py")
for f in ("lab05_model.xlsx", "lab05_financials.json", "lab05_request.txt"):
    urllib.request.urlretrieve(f"{REPO}/labs/{f}", f)

sys.modules.pop("fina4030a", None)
import fina4030a
importlib.reload(fina4030a)

if fina4030a.__version__ < REQUIRED_CLIENT:
    print(f"!! Loaded client v{fina4030a.__version__}, needs v{REQUIRED_CLIENT}.")
    print("   Runtime > Restart session, then run this cell again.")
else:
    print(f"client v{fina4030a.__version__} loaded")

# --- your details -----------------------------------------------------------
NAME       = ""
STUDENT_ID = ""

fina4030a.configure(provider="cuhk_portal")
fina4030a.verify()


---
## The request that produced it

Read this before you open the workbook. Everything the model was told is here,
and everything it was not told is here too.


In [ ]:
print(open("lab05_request.txt", encoding="utf-8").read())


---
## The instrument: perturb, then recalculate

`openpyxl` reads a spreadsheet's **formulas** or its **stored values**, never
both, and never recomputes. That is not good enough for an audit — reading a
formula tells you what it says, not what it does.

The `formulas` package builds the dependency graph and evaluates it, so you can
change an input and watch the whole workbook move. About five seconds a run.

`recalc()` returns a **getter**. Call it with no arguments for the model as
delivered, or with cells to override:

```python
g = recalc()                                   # as delivered
g = recalc(**{"Assumptions!G6": 0.50})         # FY2026 revenue growth at 50%
g("DCF!B47")                                   # read any cell
```

Each call is independent — overrides do not persist into the next `recalc()`.


In [ ]:
import formulas, json, openpyxl

BOOK = "lab05_model.xlsx"
_XL  = formulas.ExcelModel().loads(BOOK).finish()
_B   = f"'[{BOOK}]"

def _ref(a1):
    sheet, cell = a1.split("!")
    return f"{_B}{sheet.upper()}'!{cell.upper()}"

def recalc(**changes):
    """Override zero or more cells, recalculate everything, return a getter."""
    ins = {_ref(k): v for k, v in changes.items()}
    sol = _XL.calculate(inputs=ins) if ins else _XL.calculate()
    def get(a1):
        try:
            v = sol[_ref(a1)].value
        except KeyError:
            return None                      # empty cell
        try:
            return v[0, 0]
        except Exception:
            return v
    return get

wb = openpyxl.load_workbook(BOOK)
print("tabs:", ", ".join(wb.sheetnames))
print("live formulas:",
      sum(1 for ws in wb for r in ws.iter_rows() for c in r
          if isinstance(c.value, str) and c.value.startswith("=")))

g = recalc()
print(f"\nheadline value per share: ${g('DCF!B47'):,.2f}")
print(f"WACC {g('DCF!B14'):.4%}   terminal growth {g('DCF!B17'):.2%}")


---
## First, satisfy yourself the arithmetic is right

Do not take my word for it. Recompute the historicals from the SEC extract and
the DCF headline from first principles.

If you skip this you will spend the session hunting for a wrong number and find
nothing, because there isn't one.


In [ ]:
extract = json.load(open("lab05_financials.json", encoding="utf-8"))
COLS = dict(zip("BCDEF", extract["period_ends"][::-1]))     # B=FY2021 ... F=FY2025

def src(key, end):
    row = extract["series"].get(key, {}).get(end)
    return row["val"] / 1e6 if row else None

g = recalc()
LINES = {"revenue": 6, "cogs": 8, "rd": 12, "sga": 13, "net_income": 23}

bad = []
for key, row in LINES.items():
    for col, end in COLS.items():
        wbv, exv = g(f"Income Statement!{col}{row}"), src(key, end)
        if exv is not None and abs(wbv - exv) > 0.5:
            bad.append((key, end, wbv, exv))
print(f"income statement vs SEC extract: {len(bad)} mismatches"
      f" over {len(LINES) * len(COLS)} cells")

print("\nidentities, recomputed in Python from the extract:")
for end in extract["period_ends"]:
    gp  = src("revenue", end) - src("cogs", end) - src("gross_profit", end)
    ni  = src("pretax_income", end) - src("tax_expense", end) - src("net_income", end)
    ale = src("assets", end) - src("liabilities", end) - src("equity", end)
    print(f"   {end}   rev-cogs-gp {gp:>7,.0f}   pretax-tax-ni {ni:>7,.0f}"
          f"   A-L-E {ale:>7,.0f}")

# The DCF headline, rebuilt from scratch.
fcf  = [g(f"DCF!{c}31") for c in "GHIJK"]
w, gr = g("DCF!B14"), g("DCF!B17")
pv    = sum(f / (1 + w) ** (i + 0.5) for i, f in enumerate(fcf))
tv    = fcf[-1] * (1 + gr) / (w - gr)
eq    = pv + tv / (1 + w) ** 4.5 - g("Balance Sheet!F27") + g("Balance Sheet!F6")
mine  = eq / g("Income Statement!F26")
print(f"\nDCF rebuilt independently: ${mine:,.2f}   workbook: ${g('DCF!B47'):,.2f}")
print("Note the exponent on the terminal value. Come back to it in pass 3.")


---
## Before you start: the same mistake, one layer upstream

The SEC data this model was built from was fetched by a script I wrote. The
first version of that script produced a file in which **every single number was
filed under the wrong year**.

Each fact the SEC returns carries an `fy` field. It reads as though it describes
the fact. It describes *the report the fact appeared in*. A 2025 10-K restates
two prior years of income statement and one prior balance sheet, and stamps
`fy=2025` on all of it.

Run the next cell.


In [ ]:
FACTS = [   # exactly as the SEC API returns them, $m
    {"tag": "Revenues", "val": 17519, "end": "2023-12-31", "fy": 2025},
    {"tag": "Revenues", "val": 15641, "end": "2024-12-31", "fy": 2025},
    {"tag": "Revenues", "val": 17682, "end": "2025-12-31", "fy": 2025},
]

keyed_on_fy, keyed_on_end = {}, {}
for f in FACTS:
    keyed_on_fy[f["fy"]]   = f["val"]        # three facts, one key: last one wins
    keyed_on_end[f["end"]] = f["val"]

print("keyed on `fy` (what my script did):", keyed_on_fy)
print("keyed on `end` (the fact's period):", keyed_on_end)

print("\nSo FY2025 revenue came out as", keyed_on_fy[2025],
      "- which is the 2023 figure.")
print("Flows landed two years out, stocks one year out.\n")

# And here is the check I had written to catch exactly this kind of thing.
A, L, E = 35509, 18606, 16903      # all three the 2024 balance sheet, all labelled 2025
print(f"assets - liabilities - equity = {A - L - E}")


The check passed on all five years. It passed because all three of its terms
were displaced by the same amount, so the identity survived the error it was
there to catch. Clean output, right count of years, no exceptions, wrong in
every row.

**This is not a mistake juniors make.** It is a mistake the person who wrote
this lab made, twice, while building the thing that teaches you not to. The
second time I caught it only because I went looking for a check the first one
could not have passed.

That is the whole method of today:

> **A check that has never failed is not evidence.** Find out what it would take
> to make it fail. If the answer is "nothing", it was decoration.


---
# Pass 1 — Reference integrity

The workbook's own account of itself is on the `Checks` tab. Start by reading
what it claims, then test whether it can be false.


In [ ]:
ws = wb["Checks"]
for r in range(1, 20):
    lab = ws[f"A{r}"].value
    if lab:
        print(f"  r{r:<3} {str(lab)[:62]:<62} {ws[f'L{r}'].value or ''}")


In [ ]:
CHECKS = {
    10: "balance sheet: assets = liabilities + equity",
    11: "modelled operating income = reported",
    12: "modelled pre-tax income = reported",
    13: "modelled operating cash flow = reported",
    14: "cash flow closing cash = balance sheet cash",
}

def check_gaps(get):
    """Largest absolute gap each check reports, across all ten columns."""
    out = {}
    for row, name in CHECKS.items():
        vals = [get(f"Checks!{c}{row}") for c in "BCDEFGHIJK"]
        nums = [abs(v) for v in vals if isinstance(v, (int, float))]
        out[name] = round(max(nums), 3) if nums else None
    return out

print("as delivered:")
for k, v in check_gaps(recalc()).items():
    print(f"   {v:>12}   {k}")

print("\nnow with FY2026 distributions at $20,000m instead of $5,900m:")
g = recalc(**{"Assumptions!G25": 20000})
for k, v in check_gaps(g).items():
    print(f"   {v:>12}   {k}")

print(f"\n   FY2030 closing cash  {g('Cash Flow!K24'):>12,.0f}")
print(f"   FY2030 total assets  {g('Balance Sheet!K13'):>12,.0f}")
print(f"   FY2030 total equity  {g('Balance Sheet!K22'):>12,.0f}")
print(f"   Checks tab verdict on the balance sheet: {g('Checks!L10')}")


Negative seventy-five billion dollars of cash, carried as an asset. Total assets
below zero. Status: OK.

**Your turn.** Find a change that makes one of those five checks report a
non-zero gap. Then work out what the ones that never move have in common.

Look at how the lines being checked are *defined* — click into
`Cash Flow!F8`, `Income Statement!F14`, `Income Statement!F19`, `Cash Flow!F15`.


In [ ]:
PERTURBATIONS = [
    # (label, {cell: new value}) -- one worked example, then yours.
    ("revenue growth FY2026 -> 50%", {"Assumptions!G6": 0.50}),
    ("", {}),
    ("", {}),
    ("", {}),
]

base = check_gaps(recalc())
for label, changes in PERTURBATIONS:
    if not label.strip() or not changes:
        continue
    gaps = check_gaps(recalc(**changes))
    moved = [k for k in CHECKS.values()
             if (gaps[k] or 0) > (base[k] or 0) + 0.001]
    print(f"{label}")
    print(f"    checks that moved: {moved if moved else 'NONE'}")

_done = sum(1 for l, c in PERTURBATIONS if l.strip() and c)
print(f"\n{_done} of 4 perturbations written.")


In [ ]:
PASS1 = {
    "checks_that_cannot_fail": None,   # how many of the five, as an integer
    "why_they_cannot":         "",     # one or two sentences, mechanism not adjective
    "what_did_move_a_check":   "",     # the change that worked, or "none found"
    "broken_references_found": "",     # cell refs, or state that there are none
}
_m = [k for k, v in PASS1.items() if v is None or (isinstance(v, str) and not v.strip())]
print("Pass 1 complete." if not _m else "Pass 1 still to fill: " + ", ".join(_m))


---
# Pass 2 — Hardcodes

A hardcode is a number living somewhere it cannot be found: typed inside a
formula, or typed into a cell that its neighbours compute.

Not every literal is a defect. `/365` in a working-capital conversion is a
convention. Your job is to sort the conventions from the traps, and a trap is a
number that changes the answer and that nobody would think to look for.


In [ ]:
import re

print("=== numeric literals inside formulas ===")
for sh in wb:
    for row in sh.iter_rows():
        for c in row:
            if isinstance(c.value, str) and c.value.startswith("="):
                body = re.sub(r"'[^']*'!", "", c.value)
                body = re.sub(r"\$?[A-Z]{1,2}\$?\d+", "", body)
                lits = [m for m in re.findall(r"[-+*/^(,=<>]\s*(\d+\.?\d*)", body)
                        if m not in ("0", "1")]
                if lits:
                    print(f"  {sh.title}!{c.coordinate:<6} {sorted(set(lits))!s:<14}"
                          f" {c.value[:74]}")

print("\n=== values typed into calculation ranges ===")
for sh in wb:
    if sh.title in ("Source Data", "README", "Assumptions"):
        continue
    for row in sh.iter_rows():
        for c in row:
            if isinstance(c.value, (int, float)) and c.column >= 2:
                print(f"  {sh.title}!{c.coordinate:<6} = {c.value:<8}"
                      f" {str(sh[f'A{c.row}'].value)[:52]}")


Three things in that output are worth your attention. One of them silently
freezes a real asset for five years and is the reason the model reports a
liquidity problem it does not have. Another is a threshold that decides whether
a warning appears, sitting inside the warning's own formula.

Test the ones you suspect. `recalc()` takes any cell.


In [ ]:
HARDCODES = [
    # one dict per hardcode you judge to be a defect rather than a convention
    {
        "cell":        "",
        "what":        "",   # what the number is
        "why_it_matters": "",   # what breaks, or what it hides
        "evidence":    "",   # the perturbation you ran and what it showed
    },
]

_ok = [h for h in HARDCODES
       if all(str(h.get(k, "")).strip() for k in
              ("cell", "what", "why_it_matters", "evidence"))]
print(f"{len(_ok)} hardcodes written up." if _ok else "Nothing written up yet.")


---
# Pass 3 — Missing conventions

A missing convention is a rule the model needs and does not have: no revolver,
no floor on cash, no rule for what happens when an assumption runs out of road.

Two tools. The first finds every forecast line that never changes. The second
nudges each DCF input in turn and reports what the headline does — which is how
you find an input that looks live and is not.


In [ ]:
print("=== forecast lines that are constant across FY2026E-FY2030E ===")
g = recalc()
for sh_name in ("Assumptions", "Balance Sheet", "Income Statement", "Cash Flow"):
    for r in range(1, wb[sh_name].max_row + 1):
        lab = wb[sh_name][f"A{r}"].value
        if not lab:
            continue
        vals = [g(f"{sh_name}!{c}{r}") for c in "GHIJK"]
        nums = [v for v in vals if isinstance(v, (int, float))]
        if len(nums) == 5 and max(nums) - min(nums) < 1e-9 and abs(nums[0]) > 1e-9:
            print(f"  {sh_name}!{r:<3} {str(lab).strip()[:52]:<52} {nums[0]:>12,.1f}")


In [ ]:
INPUTS = {
    "DCF!B5":  "risk-free rate",
    "DCF!B6":  "equity risk premium",
    "DCF!B7":  "levered beta",
    "DCF!B9":  "pre-tax cost of debt",
    "DCF!B12": "target weight - equity",
    "DCF!B17": "terminal growth rate",
    "DCF!B18": "exit EV/EBITDA multiple",
    "DCF!B19": "mid-year convention",
    "DCF!B20": "short-term investments in the bridge",
}

base = recalc()
b0 = base("DCF!B47")
print(f"{'input':<38}{'from':>9}{'to':>9}{'per share':>22}")
for cell, label in INPUTS.items():
    v0 = base(cell)
    v1 = v0 * 1.1 if v0 else 1          # nudge it, or switch it on if it is zero
    ps = recalc(**{cell: v1})("DCF!B47")
    print(f"  {label:<36}{v0:>9,.3f}{v1:>9,.3f}"
          f"   ${b0:8,.2f} -> ${ps:8,.2f}   ({ps - b0:+.2f})")


One row of that table should stop you. Read the label of the input above it,
then read its own label, then look at what the workbook does with it
(`DCF!B44`). Then work out what the correct number would be and what it is
worth per share.


In [ ]:
PASS3 = [
    # one dict per missing convention
    {
        "what_is_missing": "",
        "where":           "",     # cell or tab
        "consequence":     "",     # what the model does instead, concretely
        "worth_per_share": None,   # if you can size it, in dollars; else None
    },
]

_ok = [p for p in PASS3 if str(p.get("what_is_missing", "")).strip()]
print(f"{len(_ok)} written up." if _ok else "Nothing written up yet.")


---
# Pass 4 — Unverified inputs

Every number in the workbook came from one of three places: the SEC extract, a
calculation, or somebody's head. The third category is not illegitimate — a DCF
cannot be built without judgement — but it has to be **visible**, and here it is
not, because there is no source column anywhere in the file.

Build one. For each input below, say where the number came from. The extract
records which XBRL tag supplied each data line, so those you can check; the rest
you cannot, and that is the finding.


In [ ]:
print("what the extract can vouch for:")
for k, tag in sorted(extract["tags_used"].items()):
    print(f"   {k:<20} {tag}")
print(f"\nnot found in the filings at all: {extract['not_found'] or 'none'}")
print("\nEvery number NOT in that list came from somewhere else. Where?")


In [ ]:
PROVENANCE = {
    # cell -> where the number came from, in your words.
    # "SEC extract", "derived from X", "the model's judgement, unsourced", ...
    "DCF!B5  risk-free rate 4.3%":        "",
    "DCF!B6  equity risk premium 5.0%":   "",
    "DCF!B7  levered beta 1.05":          "",
    "DCF!B9  pre-tax cost of debt 5.0%":  "",
    "DCF!B12 target equity weight 90%":   "",
    "DCF!B17 terminal growth 2.5%":       "",
    "DCF!B18 exit multiple 12.0x":        "",
    "Assumptions!G11 tax rate 14.0%":     "",
    "Assumptions!G22 other non-cash 2.0% of revenue": "",
}

_blank = [k for k, v in PROVENANCE.items() if not str(v).strip()]
print(f"{len(PROVENANCE) - len(_blank)} of {len(PROVENANCE)} traced.")
if not _blank:
    unsourced = sum(1 for v in PROVENANCE.values() if "unsourced" in v.lower())
    print(f"You judged {unsourced} of them unsourced.")
    print("Together, how much of the final answer do those set?")


**And the harder half of this pass: check for fabrication.**

The forecast needs FY2020 opening balances for its working-capital and PP&E
roll-forwards. Look at `Source Data` rows 36–45. The extract's headline
`period_ends` covers FY2021–25 only.

So either the model found FY2020 somewhere, or it made it up. Find out which.
Do not assume the answer — the whole point of this pass is that "it looks
plausible" is not a finding either way.


In [ ]:
FY2020 = {"cash": 38, "receivables": 39, "inventory": 40, "payables": 41,
          "ppe_net": 42, "lt_debt": 43, "st_debt": 44, "equity": 45}

print(f"{'line':<14}{'model':>10}{'extract 2020-12-31':>22}{'match':>8}")
for key, row in FY2020.items():
    mv = wb["Source Data"][f"B{row}"].value
    ev = src(key, "2020-12-31")
    ok = ev is not None and abs(ev - mv) < 0.51
    shown = f"{ev:,.0f}" if ev is not None else "NOT IN EXTRACT"
    print(f"{key:<14}{mv:>10,.0f}{shown:>22}{('yes' if ok else 'NO'):>8}")


---
## What your worst defect is worth

Pick the one defect you would actually raise if this landed on your desk, and
put a number on it. Rebuild the affected part correctly with `recalc()` and
report the corrected value per share.

"Material" is not a number. `$4.12` is a number.


In [ ]:
def per_share(**changes):
    return recalc(**changes)("DCF!B47")

BASE = per_share()
print(f"as delivered: ${BASE:,.2f}")

# --- your correction here ---------------------------------------------------
WORST = {
    "what":            "",     # the defect, one sentence
    "where":           "",     # cell or tab
    "class":           "",     # reference integrity | hardcode | missing convention | unverified input
    "how_you_fixed_it": "",    # the change you made and why it is the right one
    "corrected":       None,   # per share, after your correction
}

if WORST["corrected"] is not None:
    d = WORST["corrected"] - BASE
    print(f"corrected:    ${WORST['corrected']:,.2f}   ({d:+,.2f}, {d / BASE:+.1%})")
_m = [k for k, v in WORST.items()
      if v is None or (isinstance(v, str) and not v.strip())]
print("Complete." if not _m else "Still to fill in: " + ", ".join(_m))


---
## Now ask the model

You have spent an hour on this. Give the same job to the system that built it
and see what it finds — specifically, whether it can identify a check that
cannot fail.

One call. The question is deliberately the one you were asked at the top, and
the prompt hands it the definitions of every line being checked, so it has
strictly more to work with than you started with.

**Then mark its answer.** Take any check it says can fail and do the algebra:
substitute the definitions it was given into one another and see whether the
terms cancel. Two lines settle it.

For example, the prompt tells it both of these:

```
Cash Flow!F8   other non-cash = reported CFO - F6 - F7 - (F9+F10+F11)
Cash Flow!F12  modelled CFO   = SUM(F6:F11)
```

Substitute the first into the second. What is left?

Watch particularly for an answer that names the right condition and then does
not test it — "not an identity *unless* the line is hard-linked to the source"
is only useful if you go and look at whether it is.


In [ ]:
rows = []
for r in range(4, 20):
    lab = wb["Checks"][f"A{r}"].value
    if lab:
        rows.append(f"row {r}: {str(lab).strip()}  [status: {wb['Checks'][f'L{r}'].value}]"
                    f"  formula: {wb['Checks'][f'L{r}'].value and wb['Checks'][f'B{r}'].value}")

PROMPT = """Below is the integrity-check block from a three-statement financial
model in Excel, together with how each checked line is defined elsewhere in the
workbook.

For each check, answer one question: what would have to change in the model for
this check to report a failure? If the answer is "nothing" — if the check is an
arithmetic identity that holds by construction — say so plainly and explain the
mechanism.

Do not comment on whether the model is good. Only on whether each check is
capable of being false.

CHECK BLOCK:
""" + "\n".join(rows) + """

HOW THE CHECKED LINES ARE DEFINED:
  Cash Flow!F8   Other non-cash and deferred items = 'Source Data'!F21 - F6 - F7 - (F9+F10+F11)
  Cash Flow!F12  Cash flow from operations = SUM(F6:F11)
  Cash Flow!F15  Net purchases/maturities of investments = (BS!F6 - BS!E6) - F12 - F14 - F20
  Income Statement!F14  Other operating expense/(income) = F9 - F12 - F13 - 'Source Data'!F12
  Income Statement!F19  Other income/(expense) net = 'Source Data'!F14 - F15 + F18
  Balance Sheet!G6      Cash = 'Cash Flow'!G24  (the closing line of the cash flow statement)
"""

audit = fina4030a.complete(PROMPT, temperature=0.0, max_tokens=1200)
print(audit)


In [ ]:
MODEL_AUDIT = {
    "checks_it_called_unfalsifiable": None,  # integer
    "checks_that_ARE_unfalsifiable":  None,  # integer -- your answer, from pass 1
    "did_it_agree_with_you":          "",    # where it did and did not
    "anything_it_found_you_missed":   "",    # or "nothing"
    "one_it_got_wrong_and_the_algebra": "",  # name a check, show the substitution
}
_m = [k for k, v in MODEL_AUDIT.items()
      if v is None or (isinstance(v, str) and not v.strip())]
print("Complete." if not _m else "Still to fill in: " + ", ".join(_m))


---
## Findings


In [ ]:
FINDINGS = {
    "classes_found":        "",   # which of the four, and how many instances each
    "class_absent":         "",   # any you concluded was absent, and how you showed it
    "worst_defect":         "",   # one line, with the per-share number
    "what_review_misses":   "",   # would your firm's actual review have caught it?
    "the_check_you_would_add": "",  # one check this model should carry and does not
    "confidence":           None, # 1-5
}

_m = [k for k, v in FINDINGS.items()
      if v is None or (isinstance(v, str) and not v.strip())]
print("Complete." if not _m else "Still to fill in: " + ", ".join(_m))


In [ ]:
n_formulas = sum(1 for ws in wb for r in ws.iter_rows() for c in r
                 if isinstance(c.value, str) and c.value.startswith("="))
n_perturb  = len([p for p in PERTURBATIONS if p[0].strip() and p[1]])
periods    = ", ".join(extract["period_ends"])

print(fina4030a.appendix(
    student=f"{NAME} ({STUDENT_ID})",
    verification=FINDINGS.get("worst_defect", ""),
    residual_risk=FINDINGS.get("what_review_misses", ""),
    reproducibility=(
        f"Audit of lab05_model.xlsx ({n_formulas} live formulas), recalculated "
        f"with the `formulas` package. Source extract: SEC companyfacts, CIK "
        f"{extract['cik']}, periods {periods}. {n_perturb} perturbations run."),
))

fina4030a.save_transcript("lab05_transcript.json")


---

## What to submit

1. This notebook with outputs intact.
2. `lab05_transcript.json`.
3. The appendix printed above.

## One thing to carry forward

You audited a model whose arithmetic was correct throughout. Every number in it
was right. It still could not be trusted, because the parts of it that claimed
to have checked something had not — they had restated it.

That distinction is the whole job:

- A **calculation** turns inputs into an output.
- A **check** is a calculation that can come out wrong when something is wrong.

A check built from the same terms as the thing it checks is the first kind
wearing the clothes of the second. It costs a formula, it looks like diligence,
and it certifies nothing. You will meet a great many of these, and from now on
you have a way to tell: *what would have to change for this to fail?*

Then turn it around. The four passes you ran are a **specification for what a
model must be able to prove about itself** — and that specification is the
beginning of your Class 9 governance dossier.

---

### Next

**Class 6** is portfolio construction, where the failure is quieter than this
one. The optimiser's arithmetic is right, its efficient frontier is clean, and
it rests on a covariance matrix that should never have been inverted. Nothing
flags it — and by then you will know what to ask of the thing that didn't.
